### Library loadings

In [1]:
#library loading goes here

from time import time
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import re
from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score,classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

### Category definition and fetching+cleaning 20newsgroups

In [2]:

categories = [
    "alt.atheism",
    "comp.graphics",
    "comp.os.ms-windows.misc",
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.windows.x",
    "misc.forsale",
    "rec.autos",
    "rec.motorcycles",
    "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.crypt",
    "sci.electronics",
    "sci.space",
    "soc.religion.christian",
    "talk.politics.guns",
    "talk.politics.mideast",
    "talk.politics.misc",
    "talk.religion.misc",
    "sci.med"
]

data_all = fetch_20newsgroups(subset="all",categories=categories)
data_all2 = fetch_20newsgroups(subset="all",  remove=('footers','quotes'), categories=categories)
cleanedata = []
headerpattern = r'(?i)^(From:|Subject:|Date:|Distribution:|Organization:|Lines:|Nntp-Posting-Host:).*?(?:\n\s*.*?)*\n*'

start_time = time()
for doc in data_all2.data:
    cleaned=re.sub(headerpattern,'', doc, flags=re.MULTILINE) # removing header
    cleaned = re.sub(r'\n{2,}', '\n', cleaned)
    cleanedata.append(cleaned.strip())
end_time = time()

time_remove_header = end_time - start_time

X = cleanedata
y = data_all2.target

#input train, input test, output train and output test defined below
inp_train, inp_test, out_train, out_test = train_test_split(X, y, test_size=0.5, random_state=42)

names_data_labels = data_all2.target_names # contains names like 'talk.politics.guns'
print("Names of categories contained in the data")
print("\n")
print(names_data_labels)

display(Markdown("**Sample Document from Training Data - Uncleaned:**"))
print(data_all.data[0])  # Print the first document to see what it looks like

display(Markdown("**Sample Document from Training Data - Cleaned completely:**"))
print(inp_train[0])  # Print the first document to see what it looks like

display(Markdown("**Sample Document from Testing Data:**"))
print(inp_test[0])

print(f"Time taken to remove headers: {time_remove_header:} seconds")  

verbose = True

if verbose:
    print(f"Train data has {len(inp_train)} documents ")
    print(f"Test data has {len(inp_test)} documents")

Names of categories contained in the data


['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


**Sample Document from Training Data - Uncleaned:**

From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!




**Sample Document from Training Data - Cleaned completely:**

callison@uokmax.ecn.uoknor.edu (James P. Callison)
 Re: Dealer cheated me with wrong odometer reading. Need help!
 usa
 uokmax.ecn.uoknor.edu
 Engineering Computer Network, University of Oklahoma, Norman, OK, USA
 45
This is a tricky situation; if the previous owner didn't inform
the dealer of the odometer change, then the previous owner committed
fraud, and he may be liable. The dealer may also be liable; If the
previous owner notified the dealer, or if the previous owner had the 
dash replaced at a dealer, or if the previous owner had the dash changed 
legally, any records search on the car should turn up the fact that
the odometer had been altered.  If a dealer changes the speedometer, he has
to report it (it goes into the car's service record with the manufacturer,
and on the title, if I remember correctly; the dealer told me that
the old mileage, etc. were sent to Ford when my T-Bird's speedo 
was replaced). If the odometer can be set to the old mileage, it must 
be; if it can't (

**Sample Document from Testing Data:**

shippert@cco.caltech.edu (Tim Shippert)
 Re: Infield Fly Rule
 California Institute of Technology, Pasadena
 25
 sandman.caltech.edu
	The runner can leave his base at any time.  If the ball is caught,
he's got to tag up.  If it isn't caught, he _doesn't_ have to tag up at
all.  So, if he's feeling lucky, your runner at second can sprint for glory
as soon as the ball is popped up.  If it isn't caught, he's probably scored
a run.  If it is, he's probably headed for AAA.  
	The only effect the infield fly has is to make the batter out,
thereby removing the force on the runners on base.  All other rules apply,
as if you were standing second with first open and the ball is popped up.
Time taken to remove headers: 0.5008149147033691 seconds
Train data has 9423 documents 
Test data has 9423 documents


### Count Vectorizer on Multinomial Naive Bayes

In [2]:
pipeline = make_pipeline(CountVectorizer(stop_words='english'),MultinomialNB())


start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train = end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred =end_predict-start_predict

accuracy = accuracy_score(out_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f" Time taken to train: {duration_train}") # time in seconds
print(f" Time taken to predict: {duration_pred}")# time in seconds
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0:2])

Accuracy: 0.80
 Time taken to train: 0.9783649444580078
 Time taken to predict: 0.9106519222259521
                          precision    recall  f1-score   support

             alt.atheism       0.84      0.78      0.81       377
           comp.graphics       0.58      0.83      0.68       489
 comp.os.ms-windows.misc       0.93      0.39      0.55       502
comp.sys.ibm.pc.hardware       0.56      0.84      0.67       478
   comp.sys.mac.hardware       0.92      0.57      0.70       500
          comp.windows.x       0.76      0.83      0.79       504
            misc.forsale       0.79      0.78      0.79       494
               rec.autos       0.89      0.81      0.85       500
         rec.motorcycles       0.93      0.87      0.90       486
      rec.sport.baseball       0.95      0.86      0.90       505
        rec.sport.hockey       0.84      0.96      0.90       505
               sci.crypt       0.81      0.94      0.87       498
         sci.electronics       0.87      0

### Count Vectorizer on Random Forest Classifier with estimators = 50

In [4]:

#introduced random_state so that the features selected for splits are consistent
pipeline = make_pipeline(CountVectorizer(stop_words='english'), RandomForestClassifier(n_estimators=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train = end_train-start_train


start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict-start_predict

accuracy = accuracy_score(out_test, y_pred)

print(f" Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}") 
print(f" Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

 Accuracy: 0.78
Time taken to train: 13.247355222702026
 Time taken to predict: 1.1680028438568115
                          precision    recall  f1-score   support

             alt.atheism       0.80      0.71      0.75       377
           comp.graphics       0.61      0.65      0.63       489
 comp.os.ms-windows.misc       0.66      0.82      0.74       502
comp.sys.ibm.pc.hardware       0.61      0.68      0.64       478
   comp.sys.mac.hardware       0.80      0.76      0.78       500
          comp.windows.x       0.85      0.73      0.78       504
            misc.forsale       0.76      0.83      0.79       494
               rec.autos       0.78      0.81      0.79       500
         rec.motorcycles       0.85      0.83      0.84       486
      rec.sport.baseball       0.84      0.86      0.85       505
        rec.sport.hockey       0.86      0.94      0.90       505
               sci.crypt       0.90      0.91      0.90       498
         sci.electronics       0.75      0

### Count Vectorizer on Random Forest Classifier with estimators = 100

In [5]:
#introduced random_state so that the features selected for splits are consistent
pipeline = make_pipeline(CountVectorizer(stop_words='english'), RandomForestClassifier(n_estimators=100,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train = end_train-start_train


start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict-start_predict

accuracy = accuracy_score(out_test, y_pred)

print(f" Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}") 
print(f" Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

 Accuracy: 0.79
Time taken to train: 21.568917989730835
 Time taken to predict: 1.3971030712127686
                          precision    recall  f1-score   support

             alt.atheism       0.83      0.74      0.78       377
           comp.graphics       0.65      0.69      0.67       489
 comp.os.ms-windows.misc       0.68      0.82      0.74       502
comp.sys.ibm.pc.hardware       0.65      0.68      0.67       478
   comp.sys.mac.hardware       0.81      0.79      0.80       500
          comp.windows.x       0.88      0.74      0.80       504
            misc.forsale       0.77      0.82      0.79       494
               rec.autos       0.77      0.82      0.79       500
         rec.motorcycles       0.88      0.84      0.86       486
      rec.sport.baseball       0.82      0.85      0.84       505
        rec.sport.hockey       0.85      0.95      0.90       505
               sci.crypt       0.89      0.92      0.90       498
         sci.electronics       0.77      0

### Count Vectorizer on Logistic Regression with iterations = 50

In [6]:
pipeline = make_pipeline(CountVectorizer(stop_words='english'), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.82
Time taken to train: 6.964344024658203
Time taken to predict: 0.9411211013793945
                          precision    recall  f1-score   support

             alt.atheism       0.82      0.83      0.82       377
           comp.graphics       0.71      0.71      0.71       489
 comp.os.ms-windows.misc       0.76      0.80      0.78       502
comp.sys.ibm.pc.hardware       0.67      0.71      0.69       478
   comp.sys.mac.hardware       0.79      0.81      0.80       500
          comp.windows.x       0.85      0.78      0.81       504
            misc.forsale       0.81      0.78      0.79       494
               rec.autos       0.81      0.82      0.82       500
         rec.motorcycles       0.86      0.88      0.87       486
      rec.sport.baseball       0.86      0.88      0.87       505
        rec.sport.hockey       0.88      0.92      0.90       505
               sci.crypt       0.94      0.89      0.92       498
         sci.electronics       0.71      0.75

### Count Vectorizer on Logistic Regression with iterations = 100

In [7]:
pipeline = make_pipeline(CountVectorizer(stop_words='english'), LogisticRegression(max_iter=100,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

Accuracy: 0.82
Time taken to train: 10.712165832519531
Time taken to predict: 0.8828878402709961
                          precision    recall  f1-score   support

             alt.atheism       0.82      0.83      0.83       377
           comp.graphics       0.71      0.72      0.72       489
 comp.os.ms-windows.misc       0.74      0.80      0.77       502
comp.sys.ibm.pc.hardware       0.68      0.70      0.69       478
   comp.sys.mac.hardware       0.80      0.82      0.81       500
          comp.windows.x       0.86      0.77      0.81       504
            misc.forsale       0.81      0.78      0.80       494
               rec.autos       0.80      0.82      0.81       500
         rec.motorcycles       0.86      0.90      0.88       486
      rec.sport.baseball       0.86      0.88      0.87       505
        rec.sport.hockey       0.88      0.92      0.90       505
               sci.crypt       0.95      0.89      0.92       498
         sci.electronics       0.72      0.7

### TF-IDF Vectorizer on Multinomial NB

In [8]:
# using tf-idf vectorizer
pipeline = make_pipeline(TfidfVectorizer(stop_words='english'),MultinomialNB())


start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train = end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred =end_predict-start_predict

accuracy = accuracy_score(out_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print(f" Time taken to train: {duration_train}") # time in seconds
print(f" Time taken to predict: {duration_pred}")# time in seconds
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0:2])

Accuracy: 0.81
 Time taken to train: 0.980140209197998
 Time taken to predict: 0.8771059513092041
                          precision    recall  f1-score   support

             alt.atheism       0.83      0.67      0.74       377
           comp.graphics       0.75      0.73      0.74       489
 comp.os.ms-windows.misc       0.80      0.75      0.77       502
comp.sys.ibm.pc.hardware       0.61      0.83      0.70       478
   comp.sys.mac.hardware       0.91      0.71      0.80       500
          comp.windows.x       0.88      0.84      0.86       504
            misc.forsale       0.79      0.78      0.79       494
               rec.autos       0.87      0.83      0.85       500
         rec.motorcycles       0.87      0.91      0.89       486
      rec.sport.baseball       0.93      0.89      0.91       505
        rec.sport.hockey       0.83      0.96      0.89       505
               sci.crypt       0.79      0.94      0.86       498
         sci.electronics       0.89      0.

### TF-IDF Vectorizer on Random Forest Classifier with Estimators = 50

In [9]:
#introduced random_state so that the features selected for splits are consistent
pipeline = make_pipeline(TfidfVectorizer(stop_words='english'), RandomForestClassifier(n_estimators=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train = end_train-start_train


start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict-start_predict

accuracy = accuracy_score(out_test, y_pred)

print(f" Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}") 
print(f" Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

 Accuracy: 0.78
Time taken to train: 10.226557970046997
 Time taken to predict: 1.029980182647705
                          precision    recall  f1-score   support

             alt.atheism       0.83      0.68      0.75       377
           comp.graphics       0.65      0.66      0.66       489
 comp.os.ms-windows.misc       0.67      0.79      0.72       502
comp.sys.ibm.pc.hardware       0.61      0.69      0.65       478
   comp.sys.mac.hardware       0.78      0.76      0.77       500
          comp.windows.x       0.83      0.74      0.78       504
            misc.forsale       0.74      0.83      0.78       494
               rec.autos       0.77      0.78      0.77       500
         rec.motorcycles       0.88      0.82      0.85       486
      rec.sport.baseball       0.84      0.87      0.85       505
        rec.sport.hockey       0.85      0.93      0.89       505
               sci.crypt       0.89      0.90      0.90       498
         sci.electronics       0.70      0.

### TF-IDF Vectorizer on Random Forest Classifier with Estimators = 100

In [10]:
#introduced random_state so that the features selected for splits are consistent
pipeline = make_pipeline(TfidfVectorizer(stop_words='english'), RandomForestClassifier(n_estimators=100,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train = end_train-start_train


start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict-start_predict

accuracy = accuracy_score(out_test, y_pred)

print(f" Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}") 
print(f" Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil= pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

 Accuracy: 0.79
Time taken to train: 21.635447025299072
 Time taken to predict: 1.4498388767242432
                          precision    recall  f1-score   support

             alt.atheism       0.84      0.69      0.76       377
           comp.graphics       0.68      0.68      0.68       489
 comp.os.ms-windows.misc       0.68      0.80      0.74       502
comp.sys.ibm.pc.hardware       0.64      0.71      0.68       478
   comp.sys.mac.hardware       0.80      0.78      0.79       500
          comp.windows.x       0.83      0.76      0.79       504
            misc.forsale       0.77      0.83      0.80       494
               rec.autos       0.80      0.80      0.80       500
         rec.motorcycles       0.89      0.84      0.86       486
      rec.sport.baseball       0.84      0.87      0.86       505
        rec.sport.hockey       0.86      0.94      0.90       505
               sci.crypt       0.90      0.91      0.91       498
         sci.electronics       0.73      0

### TF-IDF Vectorizer on Logistic Regression with Iterations = 50

In [11]:

pipeline = make_pipeline(TfidfVectorizer(stop_words='english'), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])


/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.84
Time taken to train: 6.567218065261841
Time taken to predict: 0.8993480205535889
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.81      0.83       377
           comp.graphics       0.72      0.77      0.74       489
 comp.os.ms-windows.misc       0.77      0.80      0.79       502
comp.sys.ibm.pc.hardware       0.71      0.74      0.73       478
   comp.sys.mac.hardware       0.84      0.78      0.81       500
          comp.windows.x       0.85      0.81      0.83       504
            misc.forsale       0.82      0.81      0.81       494
               rec.autos       0.86      0.85      0.86       500
         rec.motorcycles       0.86      0.90      0.88       486
      rec.sport.baseball       0.91      0.91      0.91       505
        rec.sport.hockey       0.86      0.95      0.90       505
               sci.crypt       0.98      0.90      0.94       498
         sci.electronics       0.79      0.79

### TF-IDF Vectorizer on Logistic Regression with Iterations = 100

In [12]:
pipeline = make_pipeline(TfidfVectorizer(stop_words='english'), LogisticRegression(max_iter=100,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

Accuracy: 0.84
Time taken to train: 6.786263704299927
Time taken to predict: 0.8647279739379883
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.81      0.83       377
           comp.graphics       0.72      0.77      0.74       489
 comp.os.ms-windows.misc       0.77      0.81      0.79       502
comp.sys.ibm.pc.hardware       0.71      0.74      0.73       478
   comp.sys.mac.hardware       0.84      0.78      0.81       500
          comp.windows.x       0.86      0.81      0.84       504
            misc.forsale       0.82      0.81      0.81       494
               rec.autos       0.86      0.85      0.86       500
         rec.motorcycles       0.86      0.90      0.88       486
      rec.sport.baseball       0.91      0.91      0.91       505
        rec.sport.hockey       0.86      0.95      0.90       505
               sci.crypt       0.98      0.90      0.94       498
         sci.electronics       0.78      0.79

### TF Vectorizer on Multinomial NB

In [23]:
pipeline1 = make_pipeline(CountVectorizer(stop_words='english'))

start_train = time()
pipeline1.fit(inp_train, out_train)
end_train = time()
duration_train1 = end_train - start_train

vectorizer = pipeline1.named_steps['countvectorizer']
count_train_matrix = vectorizer.transform(inp_train).toarray()

tf_mtx_train= np.log1p(count_train_matrix)

# Create a second pipeline for the Multinomial Naive Bayes classifier
pipeline2 = make_pipeline(MultinomialNB())

# Train the Multinomial Naive Bayes classifier
start_train = time()
pipeline2.fit(tf_mtx_train, out_train)
end_train = time()
duration_train2 = end_train - start_train

# Transform the test data to a count matrix
count_test_matrix = vectorizer.transform(inp_test).toarray()

tf_mtx_test = np.log1p(count_test_matrix)

# Make predictions
start_predict = time()
y_pred2 = pipeline2.predict(tf_mtx_test)
end_predict = time()
duration_pred2 = end_predict - start_predict

# Print the results
accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

print(f"Time taken to train model: {duration_train1 + duration_train2}")
print(f"Time taken to predict model: {duration_pred2}")
print(classification_report(out_test, y_pred2, target_names=categories))


Accuracy: 0.84
Time taken to train model: 9.877795934677124
Time taken to predict model: 5.489182949066162
                          precision    recall  f1-score   support

             alt.atheism       0.86      0.70      0.77       377
           comp.graphics       0.57      0.83      0.68       489
 comp.os.ms-windows.misc       0.93      0.45      0.60       502
comp.sys.ibm.pc.hardware       0.54      0.84      0.66       478
   comp.sys.mac.hardware       0.94      0.52      0.67       500
          comp.windows.x       0.82      0.80      0.81       504
            misc.forsale       0.79      0.78      0.79       494
               rec.autos       0.90      0.81      0.86       500
         rec.motorcycles       0.94      0.85      0.89       486
      rec.sport.baseball       0.95      0.86      0.91       505
        rec.sport.hockey       0.83      0.97      0.90       505
               sci.crypt       0.74      0.94      0.83       498
         sci.electronics       0.8

### TF Vectorizer on Logistic Regression with Iterations = 50

In [24]:
pipeline1 = make_pipeline(CountVectorizer(stop_words='english'))

start_train = time()
pipeline1.fit(inp_train, out_train)
end_train = time()
duration_train1 = end_train - start_train

vectorizer = pipeline1.named_steps['countvectorizer']
count_train_matrix = vectorizer.transform(inp_train).toarray()

tf_mtx_train= np.log1p(count_train_matrix)

pipeline2 = make_pipeline(LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline2.fit(tf_mtx_train, out_train)
end_train = time()
duration_train2 = end_train - start_train

# Transform the test data to a count matrix
count_test_matrix = vectorizer.transform(inp_test).toarray()

tf_mtx_test = np.log1p(count_test_matrix)

# Make predictions
start_predict = time()
y_pred2 = pipeline2.predict(tf_mtx_test)
end_predict = time()
duration_pred2 = end_predict - start_predict

# Print the results
accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

print(f"Time taken to train model: {duration_train1 + duration_train2}")
print(f"Time taken to predict model: {duration_pred2}")
print(classification_report(out_test, y_pred2, target_names=categories))

Accuracy: 0.84
Time taken to train model: 215.40034675598145
Time taken to predict model: 5.021062135696411
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.83      0.84       377
           comp.graphics       0.71      0.73      0.72       489
 comp.os.ms-windows.misc       0.76      0.79      0.77       502
comp.sys.ibm.pc.hardware       0.68      0.70      0.69       478
   comp.sys.mac.hardware       0.82      0.81      0.81       500
          comp.windows.x       0.85      0.79      0.82       504
            misc.forsale       0.83      0.80      0.81       494
               rec.autos       0.81      0.83      0.82       500
         rec.motorcycles       0.86      0.88      0.87       486
      rec.sport.baseball       0.87      0.91      0.89       505
        rec.sport.hockey       0.88      0.94      0.91       505
               sci.crypt       0.95      0.89      0.92       498
         sci.electronics       0.

### TF Vectorizer on Logistic Regression with Iterations = 100

In [25]:
pipeline1 = make_pipeline(CountVectorizer(stop_words='english'))

start_train = time()
pipeline1.fit(inp_train, out_train)
end_train = time()
duration_train1 = end_train - start_train

vectorizer = pipeline1.named_steps['countvectorizer']
count_train_matrix = vectorizer.transform(inp_train).toarray()

tf_mtx_train= np.log1p(count_train_matrix)

pipeline2 = make_pipeline(LogisticRegression(max_iter=100,random_state=42))

start_train = time()
pipeline2.fit(tf_mtx_train, out_train)
end_train = time()
duration_train2 = end_train - start_train

# Transform the test data to a count matrix
count_test_matrix = vectorizer.transform(inp_test).toarray()

tf_mtx_test = np.log1p(count_test_matrix)

# Make predictions
start_predict = time()
y_pred2 = pipeline2.predict(tf_mtx_test)
end_predict = time()
duration_pred2 = end_predict - start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print the results
print(f"Time taken to train model: {duration_train1 + duration_train2}")
print(f"Time taken to predict model: {duration_pred2}")
print(classification_report(out_test, y_pred2, target_names=categories))

Accuracy: 0.84
Time taken to train model: 210.31601476669312
Time taken to predict model: 5.0188658237457275
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.83      0.84       377
           comp.graphics       0.71      0.73      0.72       489
 comp.os.ms-windows.misc       0.76      0.79      0.77       502
comp.sys.ibm.pc.hardware       0.68      0.70      0.69       478
   comp.sys.mac.hardware       0.82      0.81      0.81       500
          comp.windows.x       0.85      0.79      0.82       504
            misc.forsale       0.83      0.80      0.81       494
               rec.autos       0.81      0.83      0.82       500
         rec.motorcycles       0.86      0.88      0.87       486
      rec.sport.baseball       0.87      0.91      0.89       505
        rec.sport.hockey       0.88      0.94      0.91       505
               sci.crypt       0.95      0.89      0.92       498
         sci.electronics       0

### TF Vectorizer on Random Forest Classifier with Estimators = 50

In [26]:

pipeline1 = make_pipeline(CountVectorizer(stop_words='english'))

start_train = time()
pipeline1.fit(inp_train, out_train)
end_train = time()
duration_train1 = end_train - start_train

vectorizer = pipeline1.named_steps['countvectorizer']
count_train_matrix = vectorizer.transform(inp_train).toarray()

tf_mtx_train= np.log1p(count_train_matrix)

pipeline2 = make_pipeline(RandomForestClassifier(n_estimators=50,random_state=42))

start_train = time()
pipeline2.fit(tf_mtx_train, out_train)
end_train = time()
duration_train2 = end_train - start_train

# Transform the test data to a count matrix
count_test_matrix = vectorizer.transform(inp_test).toarray()

tf_mtx_test = np.log1p(count_test_matrix)

# Make predictions
start_predict = time()
y_pred2 = pipeline2.predict(tf_mtx_test)
end_predict = time()
duration_pred2 = end_predict - start_predict
accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print the results
print(f"Time taken to train model: {duration_train1 + duration_train2}")
print(f"Time taken to predict model: {duration_pred2}")
print(classification_report(out_test, y_pred2, target_names=categories))

Accuracy: 0.84
Time taken to train model: 74.9755289554596
Time taken to predict model: 6.723430871963501
                          precision    recall  f1-score   support

             alt.atheism       0.80      0.72      0.76       377
           comp.graphics       0.62      0.65      0.64       489
 comp.os.ms-windows.misc       0.67      0.83      0.74       502
comp.sys.ibm.pc.hardware       0.62      0.69      0.65       478
   comp.sys.mac.hardware       0.81      0.77      0.79       500
          comp.windows.x       0.84      0.73      0.78       504
            misc.forsale       0.76      0.83      0.79       494
               rec.autos       0.78      0.80      0.79       500
         rec.motorcycles       0.84      0.83      0.83       486
      rec.sport.baseball       0.85      0.87      0.86       505
        rec.sport.hockey       0.87      0.94      0.90       505
               sci.crypt       0.90      0.91      0.90       498
         sci.electronics       0.75

### TF Vectorizer on Random Forest Classifier with Estimators =100

In [27]:
pipeline1 = make_pipeline(CountVectorizer(stop_words='english'))

start_train = time()
pipeline1.fit(inp_train, out_train)
end_train = time()
duration_train1 = end_train - start_train

vectorizer = pipeline1.named_steps['countvectorizer']
count_train_matrix = vectorizer.transform(inp_train).toarray()

tf_mtx_train= np.log1p(count_train_matrix)

pipeline2 = make_pipeline(RandomForestClassifier(n_estimators=100,random_state=42))

start_train = time()
pipeline2.fit(tf_mtx_train, out_train)
end_train = time()
duration_train2 = end_train - start_train

# Transform the test data to a count matrix
count_test_matrix = vectorizer.transform(inp_test).toarray()

tf_mtx_test = np.log1p(count_test_matrix)

# Make predictions
start_predict = time()
y_pred2 = pipeline2.predict(tf_mtx_test)
end_predict = time()
duration_pred2 = end_predict - start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print the results
print(f"Time taken to train model: {duration_train1 + duration_train2}")
print(f"Time taken to predict model: {duration_pred2}")
print(classification_report(out_test, y_pred2, target_names=categories))

Accuracy: 0.84
Time taken to train model: 144.0637321472168
Time taken to predict model: 8.382954120635986
                          precision    recall  f1-score   support

             alt.atheism       0.83      0.74      0.78       377
           comp.graphics       0.64      0.69      0.67       489
 comp.os.ms-windows.misc       0.69      0.83      0.75       502
comp.sys.ibm.pc.hardware       0.66      0.68      0.67       478
   comp.sys.mac.hardware       0.81      0.80      0.80       500
          comp.windows.x       0.87      0.74      0.80       504
            misc.forsale       0.77      0.82      0.80       494
               rec.autos       0.78      0.82      0.80       500
         rec.motorcycles       0.88      0.84      0.86       486
      rec.sport.baseball       0.83      0.85      0.84       505
        rec.sport.hockey       0.85      0.95      0.90       505
               sci.crypt       0.89      0.92      0.91       498
         sci.electronics       0.7

### Lowercasing = True when LR Classifier with iterations = 50 and TF-IDF feature

In [3]:
pipeline = make_pipeline(TfidfVectorizer(lowercase=True), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.83
Time taken to train: 6.671658992767334
Time taken to predict: 0.8654580116271973
                          precision    recall  f1-score   support

             alt.atheism       0.83      0.80      0.82       377
           comp.graphics       0.69      0.77      0.73       489
 comp.os.ms-windows.misc       0.77      0.78      0.78       502
comp.sys.ibm.pc.hardware       0.71      0.75      0.73       478
   comp.sys.mac.hardware       0.85      0.78      0.81       500
          comp.windows.x       0.85      0.81      0.83       504
            misc.forsale       0.77      0.82      0.79       494
               rec.autos       0.85      0.83      0.84       500
         rec.motorcycles       0.84      0.89      0.87       486
      rec.sport.baseball       0.87      0.91      0.89       505
        rec.sport.hockey       0.87      0.94      0.91       505
               sci.crypt       0.98      0.89      0.93       498
         sci.electronics       0.76      0.77

### Lowercasing = False when LR Classifier with iterations = 50 and TF-IDF feature feature

In [4]:
pipeline = make_pipeline(TfidfVectorizer(lowercase=False), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.82
Time taken to train: 7.919296026229858
Time taken to predict: 0.8682940006256104
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.80      0.82       377
           comp.graphics       0.68      0.75      0.72       489
 comp.os.ms-windows.misc       0.77      0.78      0.78       502
comp.sys.ibm.pc.hardware       0.70      0.73      0.72       478
   comp.sys.mac.hardware       0.87      0.76      0.81       500
          comp.windows.x       0.83      0.82      0.83       504
            misc.forsale       0.72      0.84      0.78       494
               rec.autos       0.85      0.83      0.84       500
         rec.motorcycles       0.83      0.90      0.86       486
      rec.sport.baseball       0.86      0.90      0.88       505
        rec.sport.hockey       0.86      0.94      0.90       505
               sci.crypt       0.98      0.88      0.93       498
         sci.electronics       0.78      0.76

### maxfeatures = 100 when LR Classifier with iterations = 50 and TF-IDF feature feature

In [5]:
pipeline = make_pipeline(TfidfVectorizer(max_features=100), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil= pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.25
Time taken to train: 1.1661338806152344
Time taken to predict: 0.769625186920166
                          precision    recall  f1-score   support

             alt.atheism       0.21      0.18      0.19       377
           comp.graphics       0.19      0.18      0.18       489
 comp.os.ms-windows.misc       0.16      0.09      0.12       502
comp.sys.ibm.pc.hardware       0.20      0.22      0.21       478
   comp.sys.mac.hardware       0.18      0.11      0.14       500
          comp.windows.x       0.26      0.30      0.28       504
            misc.forsale       0.44      0.69      0.54       494
               rec.autos       0.22      0.14      0.18       500
         rec.motorcycles       0.21      0.26      0.23       486
      rec.sport.baseball       0.26      0.31      0.28       505
        rec.sport.hockey       0.26      0.33      0.29       505
               sci.crypt       0.24      0.31      0.27       498
         sci.electronics       0.16      0.10

### maxfeatures = 200 when LR Classifier with iterations = 50 and TF-IDF feature feature

In [6]:
pipeline = make_pipeline(TfidfVectorizer(max_features=200), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil= pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n",y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.35
Time taken to train: 1.2443628311157227
Time taken to predict: 0.7884149551391602
                          precision    recall  f1-score   support

             alt.atheism       0.25      0.22      0.24       377
           comp.graphics       0.25      0.28      0.27       489
 comp.os.ms-windows.misc       0.61      0.51      0.55       502
comp.sys.ibm.pc.hardware       0.32      0.30      0.31       478
   comp.sys.mac.hardware       0.23      0.14      0.17       500
          comp.windows.x       0.40      0.38      0.39       504
            misc.forsale       0.51      0.71      0.59       494
               rec.autos       0.29      0.22      0.25       500
         rec.motorcycles       0.26      0.33      0.29       486
      rec.sport.baseball       0.33      0.35      0.34       505
        rec.sport.hockey       0.36      0.43      0.39       505
               sci.crypt       0.37      0.41      0.39       498
         sci.electronics       0.26      0.2

### stopwords = english when LR Classifier with iterations = 50 and tf-idf feature

In [7]:
pipeline = make_pipeline(TfidfVectorizer(stop_words='english'), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.84
Time taken to train: 6.300020217895508
Time taken to predict: 0.8250088691711426
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.81      0.83       377
           comp.graphics       0.72      0.77      0.74       489
 comp.os.ms-windows.misc       0.77      0.80      0.79       502
comp.sys.ibm.pc.hardware       0.71      0.74      0.73       478
   comp.sys.mac.hardware       0.84      0.78      0.81       500
          comp.windows.x       0.85      0.81      0.83       504
            misc.forsale       0.82      0.81      0.81       494
               rec.autos       0.86      0.85      0.86       500
         rec.motorcycles       0.86      0.90      0.88       486
      rec.sport.baseball       0.91      0.91      0.91       505
        rec.sport.hockey       0.86      0.95      0.90       505
               sci.crypt       0.98      0.90      0.94       498
         sci.electronics       0.79      0.79

### stopwords = None when LR Classifier with iterations = 50 and tf-idf  feature

In [8]:
pipeline = make_pipeline(TfidfVectorizer(stop_words=None), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.83
Time taken to train: 6.560438632965088
Time taken to predict: 0.8261690139770508
                          precision    recall  f1-score   support

             alt.atheism       0.83      0.80      0.82       377
           comp.graphics       0.69      0.77      0.73       489
 comp.os.ms-windows.misc       0.77      0.78      0.78       502
comp.sys.ibm.pc.hardware       0.71      0.75      0.73       478
   comp.sys.mac.hardware       0.85      0.78      0.81       500
          comp.windows.x       0.85      0.81      0.83       504
            misc.forsale       0.77      0.82      0.79       494
               rec.autos       0.85      0.83      0.84       500
         rec.motorcycles       0.84      0.89      0.87       486
      rec.sport.baseball       0.87      0.91      0.89       505
        rec.sport.hockey       0.87      0.94      0.91       505
               sci.crypt       0.98      0.89      0.93       498
         sci.electronics       0.76      0.77

### analyzer='word',ngram_range=(1, 2) when LR Classifier with iterations = 50 and tf-idf feature

In [9]:
pipeline = make_pipeline(TfidfVectorizer(analyzer='word',ngram_range=(1, 2) ), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.83
Time taken to train: 55.1867470741272
Time taken to predict: 2.2756102085113525
                          precision    recall  f1-score   support

             alt.atheism       0.85      0.80      0.83       377
           comp.graphics       0.67      0.75      0.71       489
 comp.os.ms-windows.misc       0.78      0.80      0.79       502
comp.sys.ibm.pc.hardware       0.68      0.74      0.71       478
   comp.sys.mac.hardware       0.88      0.75      0.81       500
          comp.windows.x       0.85      0.81      0.83       504
            misc.forsale       0.78      0.84      0.81       494
               rec.autos       0.85      0.84      0.85       500
         rec.motorcycles       0.86      0.89      0.87       486
      rec.sport.baseball       0.86      0.89      0.87       505
        rec.sport.hockey       0.85      0.94      0.89       505
               sci.crypt       0.97      0.91      0.94       498
         sci.electronics       0.81      0.78 

### analyzer='char',ngram_range=(1, 1) when LR Classifier with iterations = 50 and tf-idf feature

In [10]:
pipeline = make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(1, 1)), LogisticRegression(max_iter=50,random_state=42))

start_train = time()
pipeline.fit(inp_train, out_train)
end_train = time()
duration_train= end_train-start_train

start_predict = time()
y_pred = pipeline.predict(inp_test)
end_predict = time()
duration_pred = end_predict- start_predict

accuracy = accuracy_score(out_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(f"Time taken to train: {duration_train}")
print(f"Time taken to predict: {duration_pred}")
print(classification_report(out_test, y_pred, target_names=categories))

y_probabil = pipeline.predict_proba(inp_test)
print("Predicted Probabilities:\n", y_probabil[0])

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.25
Time taken to train: 1.5384268760681152
Time taken to predict: 1.2228038311004639
                          precision    recall  f1-score   support

             alt.atheism       0.27      0.12      0.17       377
           comp.graphics       0.21      0.15      0.18       489
 comp.os.ms-windows.misc       0.28      0.24      0.26       502
comp.sys.ibm.pc.hardware       0.26      0.36      0.30       478
   comp.sys.mac.hardware       0.23      0.11      0.15       500
          comp.windows.x       0.41      0.53      0.46       504
            misc.forsale       0.40      0.59      0.48       494
               rec.autos       0.23      0.14      0.17       500
         rec.motorcycles       0.23      0.21      0.22       486
      rec.sport.baseball       0.23      0.18      0.20       505
        rec.sport.hockey       0.26      0.28      0.26       505
               sci.crypt       0.27      0.47      0.34       498
         sci.electronics       0.18      0.0

/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Predicted Probabilities:
 [0.03784049 0.04722685 0.02270925 0.0342089  0.04400938 0.02305313
 0.03276333 0.05520508 0.05345961 0.06299663 0.09498053 0.07533822
 0.05554148 0.04037842 0.05629314 0.08107766 0.06129168 0.04395263
 0.04185898 0.03581462]
